# LACUNA — Results (per method)

Evaluates one unlearning run the way the paper does — **forget**, **retain**, **utility**,
**localization precision**, and **relearning leakage** — from the files each pipeline step saves.

| Step | File | Metrics |
|---|---|---|
| `eval.py` | `<run>/Panorama_SUMMARY.json` | forget / retain: EM, ES, Prob |
| `lm_eval_utility.py` | `<run>/lm_eval.json`, `<base>/intruction_tuned/lm_eval.json` | utility: ARC / HellaSwag / MMLU (+ pre-unlearning baseline) |
| `precision_metrics.py` | `<base>/cached_notebook_files/precision_metrics/{forget,unified}/<field>/<method>/metrics.json` | localization AUC (per scoring fn) |
| `extraction_leakage.py` | `<run>/extraction_leakage.json` | profiles forgotten / resurfaced after relearning |

Set the run coordinates (and `LACUNA_ARTIFACTS`), then *Run All*.

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- run coordinates (match your experiment preset) ---
# ARTIFACTS = the dir you passed to download_artifacts.py (--out / $LACUNA_ARTIFACTS)
ARTIFACTS = os.environ.get("LACUNA_ARTIFACTS", "artifacts")
SIZE   = "OLMo2-1B"        # OLMo2-1B | OLMo3-7B
REGIME = "masked"         # masked | unmasked
FIELD  = "Email_Address"  # Email_Address | Birth_City | Phone_Number | Drivers_License
METHOD = "GradientAscent"

BASE = os.path.join(ARTIFACTS, SIZE, REGIME)
RUN  = os.path.join(BASE, "unlearned_models", FIELD, METHOD)
PREC = os.path.join(BASE, "cached_notebook_files", "precision_metrics")

def load_json(p):
    return json.load(open(p)) if os.path.exists(p) else None

print("ARTIFACTS :", ARTIFACTS)
print("run dir   :", RUN)
print("exists    :", os.path.isdir(RUN))
if not os.path.isdir(RUN):
    print("\n!! Not found. Set ARTIFACTS above to the dir you passed to download_artifacts.py,")
    print("   and check SIZE/REGIME/FIELD/METHOD.")

## 1. Output-level: forget / retain / utility
Like the paper's output-level plot. **Forget** low = good forgetting; **retain** high = knowledge
preserved; **utility** (ARC/HellaSwag/MMLU) should stay near the **pre-unlearning baseline**
(dashed line).

In [ ]:
summ = load_json(os.path.join(RUN, "Panorama_SUMMARY.json"))
if summ is None:
    raise FileNotFoundError(f"No Panorama_SUMMARY.json under {RUN}. Run src/eval.py, or fix the coordinates above.")

# forget / retain (as %) — EM, ES, Prob
FR = [("EM", "exact_memorization", "retain_exact_memorization"),
      ("ES", "extraction_strength", "retain_extraction_strength"),
      ("Prob", "forget_Q_A_Prob", "retain_Q_A_Prob")]
forget = {name: 100 * summ.get(fk, float("nan")) for name, fk, rk in FR}
retain = {name: 100 * summ.get(rk, float("nan")) for name, fk, rk in FR}

# utility (as %) — lm-eval on the unlearned model + the instruction-tuned baseline
UTASKS = ["arc_challenge", "arc_easy", "hellaswag", "mmlu"]
lm      = load_json(os.path.join(RUN, "lm_eval.json")) or {}
lm_base = load_json(os.path.join(BASE, "intruction_tuned", "lm_eval.json")) or {}
util      = {t: 100 * lm[t]      for t in UTASKS if lm.get(t)      is not None}
util_base = {t: 100 * lm_base[t] for t in UTASKS if lm_base.get(t) is not None}

tbl = pd.DataFrame({"Forget": forget, "Retain": retain}).T
display(tbl.round(2))
if util:
    display(pd.DataFrame({"utility (unlearned)": util, "utility (pre-unlearning)": util_base}).round(2))
else:
    print("no lm_eval.json yet — run scripts/lm_eval_utility.py for the utility panel")

In [ ]:
# grouped bar chart: Forget | Retain | Utility, with pre-unlearning baseline dashed lines
GCOL = {"Forget": "#4c72b0", "Retain": "#55a868", "Utility": "#c44e52"}
groups = [("Forget", forget, None), ("Retain", retain, None), ("Utility", util, util_base)]
labels, vals, base_vals, seps, bar_colors = [], [], [], [], []
for gname, d, b in groups:
    if not d:
        continue
    seps.append((len(labels), len(labels) + len(d), gname))
    for k in d:
        labels.append(k); vals.append(d[k]); base_vals.append((b or {}).get(k))
        bar_colors.append(GCOL[gname])

fig, ax = plt.subplots(figsize=(max(8, len(labels)*0.95), 4.6))
x = np.arange(len(labels))
ax.bar(x, vals, color=bar_colors, edgecolor="black", linewidth=1.2)
for xi, bv in zip(x, base_vals):        # dashed pre-unlearning utility baseline
    if bv is not None:
        ax.plot([xi-0.4, xi+0.4], [bv, bv], color="red", ls="--", lw=1.8)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylabel("Score (%)")

# headroom so the group labels sit below the title, not on top of it
ymax = max([v for v in vals if v == v] + [b for b in base_vals if b] + [1])
ax.set_ylim(0, ymax * 1.25)
for s, e, g in seps:                     # group labels (inside, near top) + separators
    ax.text((s + e - 1) / 2, ymax * 1.12, g, ha="center", fontweight="bold", fontsize=11)
    if e < len(labels):
        ax.axvline(e - 0.5, color="0.7", lw=1)
ax.set_title(f"{SIZE} — {FIELD} / {METHOD}", pad=10)
ax.text(0.995, 0.97, "red dashed = pre-unlearning utility", transform=ax.transAxes,
        ha="right", va="top", fontsize=8, color="red")
plt.tight_layout(); plt.show()

## 2. Localization precision
Per-weight ROC AUC (in-mask vs out-of-mask). The paper reports, per method, the **best AUC across
scoring families** (1.0 = perfect localization, 0.5 = indiscriminate). `forget` target = this field's
forget groups; `unified` = all injected groups.

In [ ]:
mf = load_json(os.path.join(PREC, "forget",  FIELD, METHOD, "metrics.json"))
mu = load_json(os.path.join(PREC, "unified", FIELD, METHOD, "metrics.json"))
FAMILIES = ["raw","qtile","compnorm","layernorm","signrev","reversal",
            "dirreversal","contrast","contrastnorm","eratio","contrastln","composite"]
def auc(d, m): return d.get(f"auc_{m}") if d else None
prec = pd.DataFrame([{"scoring_fn": m, "AUC (forget)": auc(mf, m), "AUC (unified)": auc(mu, m)}
                    for m in FAMILIES]).set_index("scoring_fn")
best = prec["AUC (forget)"].dropna()
if len(best):
    bfn = best.idxmax()
    print(f"BEST localization AUC (forget target): {best.max():.3f}   (scoring fn: {bfn})")
display(prec.round(4))

### 2a. ROC curves

In [ ]:
roc_path = os.path.join(PREC, "forget", FIELD, METHOD, "roc_curves.npz")
if os.path.exists(roc_path):
    z = np.load(roc_path)
    plt.figure(figsize=(5.5,5.5))
    for m in ["raw","qtile","reversal","signrev","compnorm"]:
        if f"fpr_{m}" in z:
            plt.plot(z[f"fpr_{m}"], z[f"tpr_{m}"], label=f"{m} (AUC={auc(mf,m):.3f})")
    plt.plot([0,1],[0,1],"k--",lw=1); plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title(f"ROC — {FIELD} / {METHOD}"); plt.legend(); plt.tight_layout(); plt.show()
else:
    print("no roc_curves.npz")

## 3. Relearning leakage
The resurfacing attack: after relearning on held-out PII, how many of the **forgotten** profiles
have their PII resurface (leak in any of `extraction.attempts` prompts).

In [ ]:
leak = load_json(os.path.join(RUN, "extraction_leakage.json"))
if leak:
    display(pd.Series(leak).to_frame("value"))
    fg, rs = leak["forgotten_by_unlearn"], leak["resurfaced_after_relearn"]
    plt.figure(figsize=(4,4))
    plt.bar(["forgotten\n(by unlearn)", "resurfaced\n(after relearn)"], [fg, rs],
            color=["#4c72b0", "#c44e52"], edgecolor="black")
    plt.title(f"Relearning leakage ({leak['attempts']} attempts/person)")
    plt.ylabel("# profiles"); plt.tight_layout(); plt.show()
else:
    print("run scripts/extraction_leakage.py first")

## 4. Benchmark summary

In [ ]:
summary = {
    "size": SIZE, "field": FIELD, "method": METHOD,
    "forget_ES_%": round(100*summ.get("extraction_strength", float('nan')), 2),
    "forget_Prob": summ.get("forget_Q_A_Prob"),
    "retain_ES_%": round(100*summ.get("retain_extraction_strength", float('nan')), 2),
    "retain_Prob": summ.get("retain_Q_A_Prob"),
    "utility_mean_%": round(100*np.mean([lm[t] for t in UTASKS if lm.get(t) is not None]), 2) if util else None,
    "utility_baseline_%": round(100*np.mean([lm_base[t] for t in UTASKS if lm_base.get(t) is not None]), 2) if util_base else None,
    "precision_best_AUC": round(float(best.max()), 3) if len(best) else None,
    "leak_forgotten": (leak or {}).get("forgotten_by_unlearn"),
    "leak_resurfaced": (leak or {}).get("resurfaced_after_relearn"),
}
display(pd.Series(summary).to_frame("value"))